# ಪಾಠ 04 - ಟೂಲ್ ಬಳಕೆ ವಿನ್ಯಾಸ ಮಾದರಿ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೇಮ್ವರ್ಕ್ (ಪೈಥಾನ್) ಬಳಸಿಕೊಂಡು AI ಏಜೆಂಟ್‌ಗಳಿಗಾಗಿ **ಟೂಲ್ ಬಳಕೆ** ವಿನ್ಯಾಸ ಮಾದರಿ ಕಲಿತುಕೊಳ್ಳಬಹುದು. ನಾವು ಒಳಗೊಂಡಿದ್ದೇವೆ:

- `@tool` ಅಲಂಕಾರ ಮತ್ತುTyped ಪ್ಯಾರಾಮೀಟರ್ ಗಳೊಂದಿಗೆ ಕಾರ್ಯಕ್ಷಮ ಟೂಲ್ ಗಳ ನಿರ್ಧಾರ
- ಪ್ರತಿಯೊಂದು ಟೂಲ್ ಏನು ಮಾಡುವುದನ್ನು ಮಾದರಿಗೆ ತಿಳಿಸಲು ಟೂಲ್ schemas ಅನ್ನು ಒದಗಿಸುವುದು
- `approval_mode` ಮೂಲಕ ಟೂಲ್ ಕಾರ್ಯಾಚರಣೆಯನ್ನು ನಿಯಂತ್ರಿಸುವುದು
- Pydantic ಮಾದರಿಗಳು ಮತ್ತು `response_format` ಮೂಲಕ **ರಚಿತ ಅದಾನ-ಪ್ರದಾನ** ಅನ್ನು ಹಿಂತಿರುಗಿಸುವುದು

ಸಂದರ್ಭವೆಂದರೆ **ಪ್ರವಾಸ ಬುಕ್ಕಿಂಗ್ ಏಜೆಂಟ್** ಆಗಿದ್ದು, ಗಮ್ಯಸ್ಥಾನಗಳನ್ನು ಪರಿಶೀಲಿಸುವುದು, ಲಭ್ಯತೆ ಪರಿಶೀಲಿಸುವುದು ಮತ್ತು ವಿಮಾನ ಮಾಹಿತಿ ಪಡೆಯುವುದು ಸಾಧ್ಯವಾಗುತ್ತದೆ.


## ಸ್ಥಾಪನೆ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## @tool ಅಲಂಕರಣದೊಂದಿಗೆ ಉಪಕರಣಗಳನ್ನು ವ್ಯಾಖ್ಯಾನಿಸುವುದು

`@tool` ಅಲಂಕರಣವು ಸರಳ Python ಕಾರ್ಯವನ್ನು ಎಜಂಟ್ ಕರೆಮಾಡಬಹುದಾದ ಉಪಕರಣವಾಗಿಸುತ್ತದೆ.
ಮುಖ್ಯ ಅಂಶಗಳು:

- **ಡಾಕ್‌ಸ್ಟ್ರಿಂಗ್** ಮಾದರಿಯನ್ನು ನೋಡಬಹುದಾದ ಉಪಕರಣ ವಿವರಣೆ ಆಗುತ್ತದೆ.
- **ಪ್ರಕಾರದ ಸೂಚನೆಗಳು** (`Annotated` ಸಹಿತ ವಿವರಣೆಗಳೊಂದಿಗೆ) ಉಪಕರಣ ಆಕಾರವನ್ನು ನಿರ್ಧರಿಸುತ್ತವೆ.
- `approval_mode` ಪ್ರತಿ ಕರೆನೂ ಕಾರ್ಯಗತಗೊಳಿಸುವ ಮೊದಲು ಬಳಕೆದಾರನು ಅನುಮೋದಿಸಬೇಕೇ ಎಂಬುದನ್ನು ನಿಯಂತ್ರಿಸುತ್ತದೆ.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## ಬಹು ಉಪಕರಣಗಳೊಂದಿಗೆ ಏಜೆಂಟ್ ರಚನೆ

ಬಳಕೆದಾರರ ಪ್ರಶ್ನೆಗೆ ಉತ್ತರಿಸಲು ಮಾದರಿ ಯಾವುದೇ ಉಪಕರಣಗಳನ್ನು ಕರೆಯಲು ಸಾಧ್ಯವಾಗುವಂತೆ ಎಲ್ಲಾ ಮೂರು ಉಪಕರಣಗಳನ್ನು ಕ್ಲೈಂಟ್‌ಗೆ ಪಾಸ್ಸು ಮಾಡಿ.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## ಉಪಕರಣಗಳೊಂದಿಗೆ ಸಂಕೀರ್ಣಿತ ಔಟ್‌ಪುಟ್

`response_format` ಅನ್ನು Pydantic ಮಾದರಿಯಾಗಿ ನಿಗದಿಪಡಿಸುವ ಮೂಲಕ, ಏಜೆಂಟ್ ಸ್ವತಂತ್ರ ಪಠ್ಯದ ಬದಲಾಗಿ ಒಳ್ಳೆಯ ಪ್ರಕಾರದ JSON ವಸ್ತುವನ್ನು ಮರಳಿಸಲು ಬಲವಂತಗೊಳಿಸಲಾಗಿದೆ. ಇದು ಕೆಳಗಿನ ಕೋಡ್ ಫಲಿತಾಂಶವನ್ನು ಕಾರ್ಯಕ್ರಮಾತ್ಮಕವಾಗಿ ಬಳಸಬೇಕಾಗಿದ್ದಾಗ ಉಪಯುಕ್ತವಾಗಿದೆ.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## ಸಾಧನ ಅನುಮೋದನಾ ಮಾದರಿಗಳು

`@tool` ಮೇಲೆ ಇರುವ `approval_mode` ಪರಿಮಾಣವು ಸಾಧನ ಕಾಲ್‌ಗಳು ಕಾರ್ಯಗತಿಗೊಂಡ ಮೊದಲು ಮಾನವ ಅನುಮೋದನೆಯನ್ನು ಅಗತ್ಯವಿದೆ ಎಂದು ನಿಯಂತ್ರಿಸುತ್ತದೆ:

| ಮೋಡ್ | ವರ್ತನೆ |
|---|---|
| `"never_require"` | ಸಾಧನ ಸ್ವಯಂಚಾಲಿತವಾಗಿ ಚಾಲನೆ ಹೊಂದುತ್ತದೆ — ಬಳಕೆದಾರ ದೃಢೀಕರಣ ಅಗತ್ಯವಿಲ್ಲ. |
| `"always_require"` | ಪ್ರತಿ ಕಾಲ್ ಕಾರ್ಯಗತಿಗೊಳ್ಳುವುದಕ್ಕೂ ಮೊದಲು ಬಳಕೆದಾರರಿಂದ ಅನುಮೋದನೆ ಪಡೆಯಬೇಕಾಗುತ್ತದೆ. |

ಬದಿಗಳ ಪರಿಣಾಮಗಳು ಇರುವ ಸಾಧನಗಳಿಗೆ ( ಉದಾಹರಣೆಗೆ ವಿಮಾನ ಟಿಕೆಟ್ ಬುಕ್ಕಿಂಗ್, ಕ್ರೆಡಿಟ್ ಕಾರ್ಡ್ ಚಾರ್ಜ್ ಮಾಡುವುದು) ಮಾನವನು ಪ್ರಕ್ರియೆಯಲ್ಲಿ ಇರಲು `"always_require"` ಅನ್ನು ಬಳಸಿಕೊಳ್ಳಿ.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು ಹೇಗೆ ಮಾಡುವುದು ಎಂದು ಕಲಿತಿರಿ:

1. ಟೂಲ್ ಸ್ಕೀಮಾ ಆಗಿ ಸೇವೆ ನೀಡುವ ಟೈಪ್ ಮಾಡಲಾದ ಪ್ಯಾರಾಮೀಟರ್ಸ್ ಮತ್ತು ಡಾಕ್‌ಸ್ಟ್ರಿಂಗ್‌ಗಳೊಂದಿಗೆ `@tool` ಡೆಕರೇಟರ್ ಅನ್ನು ಬಳಸಿಕೊಂಡು **ಟೂಲ್‌ಗಳನ್ನು ವ್ಯಾಖ್ಯಾನಿಸುವುದು**.
2.აგენტ್ ಸಂಕೀರ್ಣ ಪ್ರಶ್ನೆಗಳಿಗೆ ಉತ್ತರಿಸಲು ಕ್ರಮದಲ್ಲಿ ಕರೆ ಮಾಡಲು ಅನೇಕ ಟೂಲ್‌ಗಳನ್ನು **ಸಂಕಲಿಸಿ**.
3. `response_format` ಆಗಿ ಪೈಡ್ಯಾಂಟಿಕ್ ಮಾದರಿಯನ್ನು ಪಾಸ್ ಮಾಡಿ **ಸಂರಚಿತ ಅದ್ಭುತತೆ**ವನ್ನು ಮರುಹೊಂದಿಸು.
4. ಸಂವೇದನಾಶೀಲ ಕಾರ್ಯಾಚರಣೆಗಳಿಗೆ ಮಾನವನನ್ನು ಲೂಪ್ನಲ್ಲಿ ಇರಿಸುವ `approval_mode` ಬಳಸಿ **ಟೂಲ್ ಅನುಮೋದನೆಯನ್ನು ನಿಯಂತ್ರಿಸುವುದು**.

ಈ ಮಾದರಿಗಳು ಭದ್ರವಾಗಿ ಬಾಹ್ಯ ವ್ಯವಸ್ಥೆಗಳೊಂದಿಗೆ ಪರಸ್ಪರ ಕ್ರಿಯಾಶೀಲವಾಗಬಹುದಾದ ವಿಶ್ವಾಸಾರ್ಹ ಮತ್ತು ಉತ್ಪಾದನ-ತಯಾರಿಗಳನ್ನು ನಿರ್ಮಿಸುವ ಮೂಲದಲ್ಲಿವೆ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅಸ್ವೀಕಾರ**:
ಈ ದಸ್ತಾವೇಜು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಸಾಧಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ದಯವಿಟ್ಟು ಗಮನಿಸಿ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸಡ್ಡೆಗಳು ಇರಬಹುದು. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಸ್ತಾವೇಜು ಪ್ರಾಮಾಣಿಕ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದವನ್ನು ಬಳಸುವ ಮೂಲಕ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಗಳ ಅಥವಾ ತಪ್ಪು ವ್ಯಾಖ್ಯಾನಗಳ ಬಗ್ಗೆ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
